# LLE: Locally Linear Embedding

## What is LLE?

**Locally Linear Embedding (LLE)** is a nonlinear dimensionality reduction technique that, unlike PCA (which looks at global variance) and MDS (which tries to preserve *all* pairwise distances), focuses only on **local neighborhood relationships**. The core assumption behind LLE is that even though data may curve and twist globally (like a Swiss roll), if you zoom in close enough, any small neighborhood of points looks approximately **flat** — a *locally linear* patch of the underlying manifold.

LLE exploits this by:
1. Describing each point as a linear combination of its nearest neighbors (capturing the *local* geometry).
2. Finding a low-dimensional embedding where those same local relationships — the same neighbors, combined with the same weights — still hold.

Because it only has to preserve *local* structure (not global straight-line distances like MDS, or global variance like PCA), LLE can "unroll" curved manifolds far more faithfully — nearby points on the roll stay nearby in the 2D result, without being confused by unrelated points that happen to be close in raw 3D space but far apart along the actual surface.

## Steps to Compute LLE

1. **Find the k nearest neighbors** of every data point, using a standard distance metric (e.g. Euclidean) in the original high-dimensional space. `k` is a hyperparameter you choose.

2. **Construct a weight matrix W**, where each point's weights are chosen so that it can be reconstructed as accurately as possible from a linear combination of *only its k neighbors* (weights for all non-neighbors are forced to 0). This is done by minimizing a reconstruction cost:

$$
E(W) = \sum_i \left| X_i - \sum_j W_{ij} X_j \right|^2
$$

   subject to the constraint that each point's weights sum to 1:

$$
\sum_j W_{ij} = 1
$$

   Intuitively: W captures how each point sits *relative to its neighbors* — a description of local geometry that doesn't depend on the surrounding curvature.

3. **Find low-dimensional coordinates Y** that preserve those same local relationships — i.e., find points $Y_1, \ldots, Y_N$ in the target lower-dimensional space that can *still* be reconstructed from their neighbors using the *same weights W* found in step 2, by minimizing:

$$
C(Y) = \sum_i \left| Y_i - \sum_j W_{ij} Y_j \right|^2
$$

   Here, W is now fixed (already known from step 2), and the optimization solves for the best positions Y — the reverse of step 2, where W was unknown and X was fixed.

4. **Output the resulting embedding** — the solved Y coordinates form the final low-dimensional representation, one point per original sample.

## Comparison: LLE vs. PCA vs. MDS

| | PCA | MDS | LLE |
|---|---|---|---|
| Structure preserved | Directions of global maximum variance | All pairwise distances (global) | Local neighborhood relationships only |
| Linear or nonlinear | Linear | Can be linear (classical/metric MDS) or nonlinear variants | Nonlinear |
| Works on | Feature covariance matrix | Pairwise distance matrix | k-nearest-neighbor graph + local weights |
| Solution method | Eigen-decomposition (closed-form) | Iterative stress-minimization | Two-stage: weight-fitting, then eigen-decomposition of a sparse matrix |
| Handles curved manifolds well? | No — only finds flat/linear structure | Partially — distorted by straight-line vs. along-surface distance mismatch | Yes — designed specifically for this case, using only local (flat-in-the-small) geometry |
| Key hyperparameter | Number of components | Number of components | Number of neighbors (k), plus number of components |

**Expected outcome on the Swiss roll:** where PCA is expected to flatten/squash the roll from the side (losing the spiral entirely) and MDS is expected to partially unroll it (but with some distortion from preserving straight-line rather than along-surface distances), LLE is specifically designed to succeed at this exact case — since it never relies on global distances at all, only on the fact that each small neighborhood is locally flat, LLE should produce the cleanest "unrolled" 2D layout of the three, with the color gradient from `y` appearing as a smooth, non-overlapping sheet.

In [1]:
# Data manipulation
import pandas as pd # for data manipulation
import numpy as np # for data manipulation

# Visualization
import plotly.express as px # for data visualization

# Skleran
from sklearn.datasets import make_swiss_roll # for creating swiss roll
from sklearn.manifold import LocallyLinearEmbedding as LLE # for LLE dimensionality reduction
from sklearn.manifold import Isomap # for Isomap dimensionality reduction

In [12]:
# Create a swiss roll
X, y = make_swiss_roll(n_samples=2000, noise=0.05)
# Same as before: generate 2000 points on the Swiss roll's curved 2D
# surface (embedded in 3D), with y giving each point's position along
# the unrolled surface.

# Make it thinner
X[:, 1] *= .5
# Same cosmetic compression along the second coordinate as before.

# Create a flat addon to the top of the swiss roll
X_x=np.zeros((300,1))
# Create 300 points where the FIRST coordinate (x) is fixed at exactly
# 0 for all of them — shape (300, 1), a column of zeros. This means
# this new patch of points will lie perfectly flat along one axis
# (no variation in x at all).

X_y=np.random.uniform(low=0, high=10, size=(300,1))
# Generate 300 random values for the SECOND coordinate (y), uniformly
# sampled between 0 and 10 — shape (300, 1). This gives the flat patch
# some spread/extent in one direction.

X_z=np.random.uniform(low=14, high=25, size=(300,1))
# Generate 300 random values for the THIRD coordinate (z), uniformly
# sampled between 14 and 25 — shape (300, 1). This gives the flat
# patch spread in a second direction, and — importantly — positions it
# at z values (14 to 25) well above where the Swiss roll's own z-range
# typically sits, so this new patch sits near/adjacent to the roll in
# 3D space rather than overlapping it.

X2=np.concatenate((X_x, X_y, X_z), axis=1)
# Stack the three coordinate columns (x, y, z) side by side (axis=1
# means "concatenate as new columns") to form a (300, 3) array — 300
# new 3D points that together form a flat, rectangular sheet (since x
# is constant at 0, while y and z vary freely — geometrically, this is
# a flat plane sitting at x=0).

y2=X_z.reshape(300)
# Create a "color/position" value for these 300 new points, reusing
# their z-coordinate as the label — reshaped from (300, 1) to a flat
# (300,) array to match the shape of the original y. This is just a
# convenient stand-in value for coloring the flat patch distinctly
# from the roll in later plots (it's not derived from "position along
# a manifold" the way the roll's y was, since this patch isn't rolled).

# Concatenate swiss roll and flat rectangle arrays
X_two=np.concatenate((X, X2))
# Stack the original 2000 Swiss roll points and the new 300 flat-patch
# points into a single array — default axis=0 means "stack as new
# rows" — giving X_two shape (2300, 3): one combined dataset containing
# both the curved roll AND a separate flat rectangular sheet floating
# near it.

y_two=np.concatenate((y, y2))
# Stack the corresponding color/position values the same way, giving
# y_two shape (2300,) — aligned row-for-row with X_two.

In [3]:
X.shape

(2000, 3)

In [4]:
X_two.shape

(2300, 3)

You will note that we have created two swiss rolls instead of one. The first one is standard, while the second one contains an additional rectangular addon at the top.

To see what I mean by this, let’s first define two functions that we will use throughout the project to visualize our 3D swiss rolls and the 2D embedding results.

In [13]:
## Create a 3D scatter plot
def Plot3D(X, y, plot_name):
    # A reusable version of the 3D Swiss roll plotting code from the
    # MDS notebook's opening cell, now wrapped in a function so it can
    # be called on ANY 3D array (X) with corresponding colors (y) and
    # a custom title (plot_name), instead of being hardcoded once.
    fig = px.scatter_3d(None,
                        x=X[:,0], y=X[:,1], z=X[:,2],
                        color=y,
                        height=800, width=800
                       )
    # Same 3D scatter setup as before, with height/width now explicitly
    # set to 800x800 (previously left at Plotly's default size).

    # Update chart looks
    fig.update_layout(title_text=plot_name,
                      showlegend=False,
                      legend=dict(orientation="h", yanchor="top", y=0, xanchor="center", x=0.5),
                      scene_camera=dict(up=dict(x=0, y=0, z=1),
                                            center=dict(x=0, y=0, z=-0.1),
                                            eye=dict(x=1.5, y=1.75, z=1)),
                                            margin=dict(l=0, r=0, b=0, t=0),
                      scene = dict(xaxis=dict(backgroundcolor='white',
                                              color='black',
                                              gridcolor='#f0f0f0',
                                              title_font=dict(size=10),
                                              tickfont=dict(size=10),
                                             ),
                                   yaxis=dict(backgroundcolor='white',
                                              color='black',
                                              gridcolor='#f0f0f0',
                                              title_font=dict(size=10),
                                              tickfont=dict(size=10),
                                              ),
                                   zaxis=dict(backgroundcolor='lightgrey',
                                              color='black',
                                              gridcolor='#f0f0f0',
                                              title_font=dict(size=10),
                                              tickfont=dict(size=10),
                                             )))
    # Same styling as the MDS notebook's 3D plot (camera angle, axis
    # colors, gridlines, margins), but now:
    #   - title_text=plot_name: actually shows a title this time
    #     (previously commented out), using whatever name is passed in
    #     when the function is called.
    #   - legend=dict(...) is defined here but has no visible effect,
    #     since showlegend=False is set right above it — likely leftover/
    #     unused configuration.
    #   - eye=dict(x=1.5, y=1.75, z=1) uses a slightly different default
    #     camera position/zoom than the original MDS notebook's
    #     (x=1.25, y=1.5, z=1) — a minor tweak to the initial view angle.

    # Update marker size
    fig.update_traces(marker=dict(size=3,
                                  line=dict(color='black', width=0.1)))
    fig.update(layout_coloraxis_showscale=False)
    return fig
    # Same marker styling as before, then — critically — the figure is
    # RETURNED rather than shown directly. This means calling
    # Plot3D(X, y, "some title") doesn't immediately display anything;
    # the caller must do fig = Plot3D(...); fig.show() (or similar)
    # afterward. This is what makes it reusable: you can call it
    # multiple times with different data/titles and decide separately
    # when/whether to display or further modify each result.


#----------------------------------------------
# Create a 2D scatter plot
def Plot2D(X, y, plot_name):
    # Same idea, reusable version of the 2D scatter plotting code from
    # the MDS/PCA comparison plots.
    fig = px.scatter(None, x=X[:,0], y=X[:,1],
                     labels={
                         "x": "Dimension 1",
                         "y": "Dimension 2",
                     },
                     opacity=1, color=y)
    # Same 2D scatter setup as before, with one addition: labels=dict(...)
    # explicitly renames the axis titles to "Dimension 1" and
    # "Dimension 2" — generic names appropriate here since this function
    # will be reused across several different methods (PCA, MDS, LLE,
    # Isomap), where the 2 output axes don't correspond to real physical
    # units, just abstract embedding coordinates.

    # Change chart background color
    fig.update_layout(dict(plot_bgcolor = 'white'))

    # Update axes lines
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgrey',
                     zeroline=True, zerolinewidth=1, zerolinecolor='lightgrey',
                     showline=True, linewidth=1, linecolor='black')

    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgrey',
                     zeroline=True, zerolinewidth=1, zerolinecolor='lightgrey',
                     showline=True, linewidth=1, linecolor='black')
    # Identical axis/gridline/background styling to the earlier MDS/PCA
    # 2D plots.

    # Set figure title
    fig.update_layout(title_text=plot_name)
    # Uses the passed-in plot_name instead of a hardcoded title.

    # Update marker size
    fig.update_traces(marker=dict(size=5,
                                 line=dict(color='black', width=0.3)))
    return fig
    # Same as Plot3D: returns the figure object rather than calling
    # fig.show() directly, so the caller controls when/whether to
    # display it.

In [14]:
import plotly.io as pio
pio.renderers.default = "colab"
# Set the renderer to "colab" since this is running on google colab

fig = Plot3D(X, y, "")
# Call the Plot3D function just defined, passing:
#   - X: the plain Swiss roll (2000 points) — NOT X_two, so this shows
#     the original roll alone, without the added flat sheet.
#   - y: the corresponding position-along-the-roll values, for coloring.
#   - "": an empty string for plot_name, so no title is displayed above
#     the plot.
# Since Plot3D returns the figure object rather than displaying it
# directly, this line just builds the figure and stores it in `fig` —
# nothing appears yet.

fig.show()
# Actually render and display the interactive 3D plot in the notebook
# output — this is the step that makes the figure visible, now that
# it's been constructed by the function above.

In [15]:
fig = Plot3D(X_two, y_two, "")
# Call Plot3D again, this time passing the COMBINED dataset:
#   - X_two: the 2300 points — the original 2000-point Swiss roll PLUS
#     the 300-point flat sheet added earlier.
#   - y_two: the corresponding color values for all 2300 points (the
#     roll's original y values, plus the flat sheet's z-based y2 values).
#   - "": no title.
# As before, this just builds the figure — nothing displayed yet.

fig.show()
# Render and display the 3D plot, now showing both shapes together:
# the familiar curled Swiss roll, plus the separate flat rectangular
# sheet positioned near it (at the higher z-range specified when X2
# was constructed).

Let's define two reusable functions to run LLE and Isomap algorithms

In [17]:
# Function for performing LLE and MLLE
def run_lle(num_neighbors, dims, mthd, data):
    # A reusable wrapper function around scikit-learn's LLE, so it can
    # be called repeatedly with different settings (e.g. different
    # neighbor counts, or on X vs. X_two) without repeating the setup
    # code each time.

    # Specify LLE parameters
    embed_lle = LLE(n_neighbors=num_neighbors, #
                    n_components=dims,
                    reg=0.001,
                    random_state=42,
                   )
    # Create an LLE instance:
    #   - n_neighbors=num_neighbors: the "k" from the earlier steps —
    #     how many nearest neighbors each point uses to compute its
    #     local reconstruction weights (step 1/2 of the LLE algorithm).
    #     Passed in as a parameter here, rather than hardcoded, so
    #     different values of k can be tried and compared later.
    #   - n_components=dims: the target dimensionality of the output
    #     embedding (e.g. 2, for 2D visualization) — also parameterized.
    #   - reg=0.001: a small regularization term added while solving
    #     for the reconstruction weights W (step 2). This is needed
    #     because if a point has more neighbors than the number of
    #     original dimensions, the weight-fitting problem becomes
    #     under-determined (multiple equally good solutions) or
    #     numerically unstable; a tiny regularization term stabilizes
    #     the solution without meaningfully changing it.
    #   - random_state=42: fixes any random elements in the underlying
    #     solver for reproducibility.
    #
    # Fit and transofrm the data
    result = embed_lle.fit_transform(data)
    # Run the actual LLE algorithm on the given data (the full 3-step
    # process: find neighbors, solve for weights W, then solve for the
    # low-dimensional embedding Y that preserves those weights) — same
    # fit_transform pattern used by PCA and MDS.

    # Return results
    return result
    # Return just the resulting embedding (an array of shape
    # (n_samples, dims)) — unlike Plot3D/Plot2D, which returned a
    # Plotly figure object, this returns the raw transformed data,
    # ready to be passed into Plot2D(...) for visualization afterward.

## What is `mthd`?

`mthd` is just a **parameter name** in the custom `run_lle` function — short for "method." It's not a scikit-learn thing itself; it's a variable created to *eventually* pass through to scikit-learn's real `method` parameter on the `LocallyLinearEmbedding` class.

Scikit-learn's `LocallyLinearEmbedding` actually does support a `method` argument, which selects between several **variants of LLE**, all sharing the same 3-step skeleton (neighbors → local weights → embedding) but differing in how the weights/embedding are computed to fix specific weaknesses of standard LLE:

| `method` value | Name | What it changes |
|---|---|---|
| `'standard'` | Standard LLE | The plain algorithm — can suffer from ill-conditioning when a point has more neighbors than the number of original dimensions, which regularization (`reg=0.001`) only partially compensates for. |
| `'modified'` | **MLLE** (Modified LLE) | Uses *multiple* weight vectors per neighborhood instead of one, which fixes the ill-conditioning problem more robustly than adding a small `reg` term — generally considered more reliable than standard LLE. |
| `'hessian'` | **HLLE** (Hessian LLE) | Uses the Hessian (second-derivative) operator instead of the standard reconstruction weights — better at preserving locally linear patches when the manifold isn't just locally flat but also smoothly curved, at the cost of needing more neighbors and being more computationally expensive. |
| `'ltsa'` | **LTSA** (Local Tangent Space Alignment) | A related but distinct algorithm — estimates a local tangent space at each point, then aligns all these local tangent spaces into one consistent global embedding. |

So the intent behind `mthd` in `run_lle` was almost certainly: pass `'standard'` or `'modified'` (etc.) in, and have it select the corresponding LLE variant via `method=mthd` inside the `LLE(...)` constructor. As currently written, though, that wiring is missing — `mthd` is accepted as an argument but never actually used anywhere in the function body, so every call to `run_lle` runs `'standard'` LLE regardless of what is passed for `mthd`.

Let’s create five 2D projections using different data and algorithms. Note that in all cases, we set the number of neighbors to 30 while using default values for other hyperparameters.

In [18]:
######### Regular swiss roll #########

# Standard LLE on a regular swiss roll
std_lle_res=run_lle(num_neighbors=30, dims=2, mthd='standard', data=X)
# Run LLE on the ORIGINAL Swiss roll (X, 2000 points, no flat sheet
# added), using 30 nearest neighbors and reducing to 2 dimensions.
# std_lle_res will have shape (2000, 2) — the "unrolled" 2D embedding
# of the clean roll, to be visualized and compared against the earlier
# PCA/MDS results on the same X.

######### Modified swiss roll #########

# Modified LLE on a modified swiss roll
std_mlle_res=run_lle(num_neighbors=30, dims=2, mthd='standard', data=X_two)
# Run LLE on the COMBINED dataset (X_two, 2300 points — the roll plus
# the separate flat sheet added earlier), same 30 neighbors, 2D output.
# std_mlle_res will have shape (2300, 2) — this is the run meant to
# test the "neighbor graph gets confused by a nearby unrelated surface"
# scenario flagged when X_two was first built.

In [10]:
import plotly.io as pio
pio.renderers.default = "colab"
fig = Plot2D(std_lle_res, y, 'Regular Swiss Roll - LLE')
fig

In [11]:
fig = Plot2D(std_mlle_res, y_two, 'Modified Swiss Roll - LLE')
fig